# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR² Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) Python library.

### Dataset Source
The dataset is defined by a Croissant schema accessible at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

List available record sets, their `@id`s, and associated fields and columns. This allows you to reference data entities by their unique identifiers.

### Record Sets

In [ ]:
# Explore available record sets and their fields/columns
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")

for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Number of fields: {len(rs.fields)}")
    for f in rs.fields:
        print(f"    Field: {f.name}")
        print(f"      @id: {f.id}")
        print(f"      DataType: {f.data_type}")
        print(f"      Description: {getattr(f,'description',None)}")
        if hasattr(f, 'columns') and f.columns:
            for col in f.columns:
                print(f"          Column: {col.name} (@id: {col.id})")
    print("")

### Example: View a few records (referencing by record set `@id`)

Use the `@id` of your target record set, as listed above.

In [ ]:
# Examine the first record set as an example
# Update this variable with the @id of the record set you want to explore
record_set_id = record_sets[0].id if record_sets else None

if record_set_id:
    print(f"First records from record set {record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if i>=2:  # Show only first 3 records
            break
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction
Load data from the record set(s) into DataFrames for analysis. All references use the record set and field `@id`s.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for rs_id in record_set_ids:
    records_list = list(dataset.records(record_set=rs_id))
    if records_list:
        dataframes[rs_id] = pd.DataFrame(records_list)
        print(f"Loaded DataFrame for {rs_id}: shape {dataframes[rs_id].shape}")

# Choose a main record set for further analysis (first one usually has main data)
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id and main_rs_id in dataframes:
    print(f"Columns in main record set ({main_rs_id}):\n{dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())
else:
    print("No DataFrames available to display.")

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps such as filtering, normalization, and grouping using fields and columns referenced by their `@id`s. 

You may choose one of the numeric columns for demo analysis.

In [ ]:
import numpy as np
# Try to detect a numeric field (float or int) using fields metadata

from collections import defaultdict
numeric_fields = []
group_fields = []
rs = None
for rs_ in record_sets:
    if rs_.id == main_rs_id:
        rs = rs_
        break
if rs:
    for field in rs.fields:
        if hasattr(field, 'data_type'):
            dtype = str(field.data_type).lower()
            if ('float' in dtype) or ('int' in dtype) or ('number' in dtype):
                numeric_fields.append(field.id)
            elif dtype == 'text' or 'str' in dtype or 'category' in dtype:
                group_fields.append(field.id)
else:
    print("No main record set structure found.")

# Choose the first numeric field, and a group field if available
if main_rs_id and main_rs_id in dataframes and not dataframes[main_rs_id].empty:
    numeric_field_id = None
    for f in numeric_fields:
        if f in dataframes[main_rs_id].columns:
            numeric_field_id = f
            break
    group_field_id = None
    for f in group_fields:
        if f in dataframes[main_rs_id].columns:
            group_field_id = f
            break
    if numeric_field_id:
        print(f"Using numeric field (by @id): {numeric_field_id}")
        # Demo threshold as 10. You may change as needed.
        threshold = 10
        filtered_df = dataframes[main_rs_id][dataframes[main_rs_id][numeric_field_id].astype(float) > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        colnorm = f"{numeric_field_id}_normalized"
        filtered_df[colnorm] = (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, colnorm]].head())

        # Grouping
        if group_field_id:
            print(f"Grouped data by {group_field_id} (mean of columns):")
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            display(grouped_df.head())
        else:
            print("No group field detected for grouping.")
    else:
        print("No numeric field found for demo.")
else:
    print("No main DataFrame for EDA.")

## 5. Visualization

Visualize data distributions or relationships. Here, we show an example distribution for the detected numeric field. You may adjust the field as desired.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if main_rs_id and main_rs_id in dataframes and numeric_fields:
    num_field = numeric_field_id if 'numeric_field_id' in locals() and numeric_field_id else numeric_fields[0]
    df_plot = dataframes[main_rs_id]
    if num_field in df_plot.columns:
        vals = pd.to_numeric(df_plot[num_field], errors='coerce').dropna()
        sns.histplot(vals, kde=True)
        plt.title(f"Distribution of {num_field}")
        plt.xlabel(num_field)
        plt.show()
    else:
        print(f"Field {num_field} not in DataFrame columns.")
else:
    print("No numeric field found for visualization.")

## 6. Conclusion

In this notebook, we've demonstrated how to load, explore, and process the [FAIR² Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using [`mlcroissant`](https://github.com/mlcommons/croissant). You can reference any field, column, or record set using their `@id`, as reflected throughout this notebook. For more advanced analysis, consult the Croissant schema for specific field meanings and units.